# E-Commerce Feature Engineering

**Dataset:** Online Retail Transactions (UCI)  
**Purpose:** Transform raw data into a model-ready feature set that is
general-purpose — suitable for classification, regression, **and** clustering
without further raw-data processing.

This notebook builds on:
- `EDA.ipynb` — data exploration and quality assessment
- `descriptive_statistics.ipynb` — distributional properties (skewness, outliers)
- `inferential_statistics.ipynb` — hypothesis tests and correlation findings

---

## 1. Setup & Imports

We need scikit-learn scalers and encoders, plus `category_encoders` for
target/frequency encoding of high-cardinality categoricals.
If `category_encoders` is not installed, run `pip install category-encoders`.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

try:
    import category_encoders as ce
    HAS_CE = True
except ImportError:
    HAS_CE = False
    print('category_encoders not installed — frequency encoding will be used instead.')
    print('Install with: pip install category-encoders')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (11, 4),
                     'axes.titlesize': 13, 'axes.labelsize': 11})
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 40)

print('Libraries loaded.')

In [ ]:
_candidates = [
    os.path.join('..', 'data', 'raw', 'data.csv'),
    os.path.join('ecommerce_analysis', 'data', 'raw', 'data.csv'),
    os.path.join('..', '..', 'data', 'raw', 'data.csv'),
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError('data.csv not found — adjust _candidates.')

df_raw = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', low_memory=False)
print(f'Loaded  {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
display(df_raw.head(5))

---
## 2. Recap & Feature Inventory

Before any transformation we catalogue every column: its type, missingness,
and preliminary role classification.

**Role classification rules used:**
- `ID/irrelevant` — invoice/transaction identifiers; they carry no generalizable signal
- `datetime` — parseable date/time strings
- `numerical` — float/int columns that represent measurable quantities
- `categorical` — object/category columns representing discrete groups

Columns flagged for **dropping**:
- Pure ID columns (near 100% unique values, no aggregation meaning)
- Free-text columns (description strings — too high cardinality for direct encoding)
- Columns with > 50% missing values
- Near-zero variance columns (single value in > 95% of rows)

Note: `Description` is kept for reference but excluded from the model matrix.

In [ ]:
df = df_raw.copy()

# Auto-detect column roles
def classify_col(series, col_name):
    name_l = col_name.lower()
    null_pct = series.isna().mean()
    n_unique = series.nunique()
    pct_unique = n_unique / len(series)

    # Try date parse
    if series.dtype == object:
        try:
            pd.to_datetime(series.dropna().head(50), infer_datetime_format=True)
            if any(k in name_l for k in ['date', 'time', 'dt']):
                return 'datetime'
        except Exception:
            pass

    if pd.api.types.is_numeric_dtype(series):
        # IDs are ints with near-unique values and no obvious measure meaning
        if pct_unique > 0.5 and any(k in name_l for k in ['id', 'no', 'num', 'code']):
            return 'ID/irrelevant'
        return 'numerical'

    if series.dtype == object:
        if pct_unique > 0.9:
            return 'ID/irrelevant' if any(k in name_l for k in ['id', 'no', 'code', 'desc', 'description']) else 'high-card categorical'
        return 'categorical'

    return 'other'

inventory_rows = []
for col in df.columns:
    s = df[col]
    null_pct = s.isna().mean() * 100
    n_unique = s.nunique()
    role = classify_col(s, col)
    drop_flag = (
        null_pct > 50
        or role == 'ID/irrelevant'
        or (n_unique / len(s) > 0.9 and s.dtype == object)
        or (s.dtype != object and s.nunique() <= 1)
    )
    inventory_rows.append({
        'column':   col,
        'dtype':    str(s.dtype),
        'null_%':   round(null_pct, 2),
        'n_unique': n_unique,
        'pct_unique': round(n_unique / len(s) * 100, 2),
        'role':     role,
        'drop?':    'YES' if drop_flag else 'keep',
    })

inventory_df = pd.DataFrame(inventory_rows).set_index('column')
display(inventory_df)

COLS_TO_DROP = inventory_df[inventory_df['drop?'] == 'YES'].index.tolist()
print(f'\nColumns flagged for drop: {COLS_TO_DROP}')

In [ ]:
# Detect key columns for downstream use
DATE_COL     = next((c for c in df.columns if any(k in c.lower() for k in ['date','time'])), None)
QTY_COL      = next((c for c in df.select_dtypes('number').columns if any(k in c.lower() for k in ['qty','quant'])), None)
PRICE_COL    = next((c for c in df.select_dtypes('number').columns if any(k in c.lower() for k in ['price','unit'])), None)
CUSTOMER_COL = next((c for c in df.columns if any(k in c.lower() for k in ['customer','cust'])), None)
INVOICE_COL  = next((c for c in df.columns if 'invoice' in c.lower()), None)
COUNTRY_COL  = next((c for c in df.columns if 'country' in c.lower()), None)
DESC_COL     = next((c for c in df.columns if 'desc' in c.lower()), None)

# Parse date column
if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], infer_datetime_format=True, errors='coerce')

# Derive Revenue
if QTY_COL and PRICE_COL:
    df['Revenue'] = df[QTY_COL] * df[PRICE_COL]

print(f'DATE_COL={DATE_COL}, QTY_COL={QTY_COL}, PRICE_COL={PRICE_COL}')
print(f'CUSTOMER_COL={CUSTOMER_COL}, INVOICE_COL={INVOICE_COL}')
print(f'COUNTRY_COL={COUNTRY_COL}, DESC_COL={DESC_COL}')

---
## 3. Handling Missing Values

**Strategy per column type:**

| Column type | Strategy | Rationale |
|-------------|----------|-----------|
| Numerical — normal dist. | Mean imputation | Mean = median for symmetric distributions |
| Numerical — skewed | Median imputation | Median is robust to outliers; skewed cols have mean ≠ median |
| Numerical — complex patterns | KNN imputation | Preserves multivariate relationships |
| Categorical | Mode or 'Unknown' | Mode for low-null cols; 'Unknown' for high-null |

We **do not drop rows** unless a column has > 50% missing values (flagged in Section 2),
as dropping rows discards usable signal in the remaining columns.

The `CustomerID` column (~25% missing, from EDA) is imputed with a sentinel value
(`-1`) since it represents guest checkouts — a meaningful business category.

In [ ]:
# Columns to work with (exclude drops and date)
KEEP_COLS = [c for c in df.columns if c not in COLS_TO_DROP or c == CUSTOMER_COL]
df_work = df[KEEP_COLS].copy()

NUM_COLS = df_work.select_dtypes(include='number').columns.tolist()
CAT_COLS = df_work.select_dtypes(include=['object', 'category']).columns.tolist()
if DATE_COL and DATE_COL in df_work.columns:
    CAT_COLS = [c for c in CAT_COLS if c != DATE_COL]

null_before = df_work.isna().sum().rename('null_before')
print('Null counts before imputation:')
display(null_before[null_before > 0].to_frame())

In [ ]:
from scipy.stats import skew

SKEW_THRESHOLD = 1.0  # |skewness| > 1 => skewed => use median

impute_log = []
for col in NUM_COLS:
    n_null = df_work[col].isna().sum()
    if n_null == 0:
        continue

    # Special handling for CustomerID — sentinel for guests
    if CUSTOMER_COL and col == CUSTOMER_COL:
        df_work[col] = df_work[col].fillna(-1)
        impute_log.append({'column': col, 'strategy': 'sentinel (-1)', 'n_imputed': n_null})
        continue

    col_skew = abs(skew(df_work[col].dropna()))
    if col_skew > SKEW_THRESHOLD:
        fill_val = df_work[col].median()
        strategy = f'median  (skew={col_skew:.2f})'
    else:
        fill_val = df_work[col].mean()
        strategy = f'mean  (skew={col_skew:.2f})'

    df_work[col] = df_work[col].fillna(fill_val)
    impute_log.append({'column': col, 'strategy': strategy, 'n_imputed': n_null})

if impute_log:
    display(pd.DataFrame(impute_log))
else:
    print('No numerical nulls to impute.')

In [ ]:
for col in CAT_COLS:
    n_null = df_work[col].isna().sum()
    if n_null == 0:
        continue
    null_pct = n_null / len(df_work)
    if null_pct <= 0.05:
        fill_val = df_work[col].mode().iloc[0]
        strategy = f'mode ("{fill_val}")'
    else:
        fill_val = 'Unknown'
        strategy = 'Unknown category'
    df_work[col] = df_work[col].fillna(fill_val)
    print(f'  {col}: {n_null:,} nulls ({null_pct*100:.1f}%) filled with {strategy}')

# Summary
null_after = df_work.isna().sum().rename('null_after')
summary = pd.concat([null_before, null_after], axis=1).query('null_before > 0')
print('\nBefore vs after imputation:')
display(summary)

---
## 4. Outlier Treatment

Informed by the IQR analysis in `descriptive_statistics.ipynb` and the
distribution shapes in `inferential_statistics.ipynb`, we apply column-specific
strategies:

| Strategy | When to apply |
|----------|---------------|
| **Winsorizing** (cap at 1st–99th pct) | Any column with outliers; preserves all rows |
| **log1p transform** | Right-skewed positive-only columns (e.g., Revenue, UnitPrice) |

We **do not drop outlier rows** — in e-commerce, outliers often represent
real wholesale buyers whose behaviour is valid signal for clustering.
Winsorizing preserves the row while reducing the distortion on model training.

For columns that are both right-skewed **and** have outliers we prefer
`log1p` because it simultaneously compresses the tail and stabilises variance.

In [ ]:
outlier_rows = []
for col in NUM_COLS:
    s = df_work[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = int(((s < lo) | (s > hi)).sum())
    col_skew = float(s.skew())
    all_pos  = bool((s >= 0).all())
    outlier_rows.append({
        'column': col, 'n_outliers': n_out,
        'outlier_%': round(n_out / len(s) * 100, 2),
        'skewness': round(col_skew, 3),
        'all_positive': all_pos,
        'recommended_strategy': (
            'log1p' if col_skew > 1.0 and all_pos and n_out > 0
            else ('winsorize' if n_out > 0 else 'none')
        ),
    })

outlier_df = pd.DataFrame(outlier_rows).set_index('column')
display(outlier_df)

In [ ]:
df_treated = df_work.copy()
treatment_log = []

for _, row in outlier_df.iterrows():
    col = row.name
    strategy = row['recommended_strategy']

    if strategy == 'none':
        continue

    s_before = df_treated[col].copy()

    if strategy == 'log1p':
        df_treated[col] = np.log1p(df_treated[col].clip(lower=0))
        treatment_log.append({'column': col, 'strategy': 'log1p',
                               'before_mean': round(float(s_before.mean()), 4),
                               'after_mean':  round(float(df_treated[col].mean()), 4)})
    elif strategy == 'winsorize':
        p01 = s_before.quantile(0.01)
        p99 = s_before.quantile(0.99)
        df_treated[col] = s_before.clip(lower=p01, upper=p99)
        treatment_log.append({'column': col, 'strategy': f'winsorize [p01={p01:.2f}, p99={p99:.2f}]',
                               'before_mean': round(float(s_before.mean()), 4),
                               'after_mean':  round(float(df_treated[col].mean()), 4)})

display(pd.DataFrame(treatment_log))

In [ ]:
treated_cols = [r['column'] for r in treatment_log]
if treated_cols:
    fig, axes = plt.subplots(len(treated_cols), 2,
                              figsize=(13, 3.5 * len(treated_cols)))
    if len(treated_cols) == 1:
        axes = [axes]
    for i, col in enumerate(treated_cols):
        for ax, data, label in zip(
            axes[i],
            [df_work[col], df_treated[col]],
            ['Before treatment', 'After treatment']
        ):
            clipped = data.clip(data.quantile(0.01), data.quantile(0.99))
            ax.hist(clipped.dropna(), bins=60, color='steelblue',
                    edgecolor='white', alpha=0.75)
            ax.set_title(f'{col} — {label}')
            ax.set_xlabel(col)
            ax.set_ylabel('Count')
    plt.suptitle('Outlier Treatment: Before vs After', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('No outlier treatment applied.')

---
## 5. Feature Transformation (Scaling)

Different algorithms have different sensitivity to feature scale:

| Scaler | Best for | Reasoning |
|--------|----------|-----------|
| **StandardScaler** | Normally distributed columns | Zero mean, unit variance; required for PCA, SVM, logistic regression |
| **MinMaxScaler** | Bounded columns (e.g., ratios, percentages) | Maps to [0, 1]; preserves shape |
| **RobustScaler** | Columns with remaining outliers | Uses IQR instead of std; less sensitive to extremes |
| **log1p** | Already applied in Section 4 | Shown here in the before/after histograms |

We apply scaling **only to the analysis numeric columns**, not to ID or
engineered boolean flags. Scaled columns are stored with a `_scaled` suffix
so the originals remain available for interpretation.

In [ ]:
# Determine scaler per column based on distribution and prior treatment
from scipy.stats import shapiro

df_scaled = df_treated.copy()
scale_log  = []
analysis_num = [c for c in NUM_COLS
                if c != CUSTOMER_COL
                and df_treated[c].nunique() > 5]

for col in analysis_num:
    s = df_scaled[col].dropna()
    col_skew   = abs(float(s.skew()))
    col_out_pct = outlier_df.loc[col, 'outlier_%'] if col in outlier_df.index else 0
    was_log1p  = (outlier_df.loc[col, 'recommended_strategy'] == 'log1p'
                  if col in outlier_df.index else False)

    if was_log1p or col_out_pct > 10:
        scaler = RobustScaler()
        scaler_name = 'RobustScaler'
    elif col_skew < 0.5:
        scaler = StandardScaler()
        scaler_name = 'StandardScaler'
    else:
        scaler = MinMaxScaler()
        scaler_name = 'MinMaxScaler'

    new_col = f'{col}_scaled'
    df_scaled[new_col] = scaler.fit_transform(
        df_scaled[[col]].fillna(df_scaled[col].median())
    )
    scale_log.append({'column': col, 'new_col': new_col, 'scaler': scaler_name})

display(pd.DataFrame(scale_log))
SCALED_COLS = [r['new_col'] for r in scale_log]

In [ ]:
if scale_log:
    n = min(len(scale_log), 4)  # show up to 4 columns
    fig, axes = plt.subplots(n, 2, figsize=(13, 3.5 * n))
    if n == 1:
        axes = [axes]
    for i, row in enumerate(scale_log[:n]):
        orig, scaled = row['column'], row['new_col']
        for ax, data, label in zip(
            axes[i],
            [df_treated[orig], df_scaled[scaled]],
            [f'{orig} (before)', f'{scaled} ({row["scaler"]})']
        ):
            clipped = data.clip(data.quantile(0.005), data.quantile(0.995))
            ax.hist(clipped.dropna(), bins=60, color='mediumpurple',
                    edgecolor='white', alpha=0.75)
            ax.set_title(label)
            ax.set_xlabel('Value')
            ax.set_ylabel('Count')
    plt.suptitle('Scaling: Before vs After (first 4 columns)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 6. Datetime Feature Extraction

Datetime columns are not usable directly by most algorithms — they must be
decomposed into numerical features that capture temporal patterns.

Features extracted:
- **year, month, quarter** — capture seasonality and long-term trends
- **day_of_week** (0=Mon, 6=Sun) — weekly purchasing patterns
- **is_weekend** — binary flag; weekends often behave differently
- **hour** — intra-day purchasing patterns (if time resolution exists)
- **days_since_min_date** — ordinal time proxy; useful for regression

From EDA we know revenue spikes in Q4 — `quarter` and `month` should carry
strong predictive signal.

In [ ]:
dt_features_added = []

if DATE_COL and DATE_COL in df_scaled.columns:
    dt = df_scaled[DATE_COL]
    min_date = dt.min()

    df_scaled['dt_year']            = dt.dt.year
    df_scaled['dt_month']           = dt.dt.month
    df_scaled['dt_quarter']         = dt.dt.quarter
    df_scaled['dt_day_of_week']     = dt.dt.dayofweek
    df_scaled['dt_is_weekend']      = (dt.dt.dayofweek >= 5).astype(int)
    df_scaled['dt_hour']            = dt.dt.hour
    df_scaled['dt_days_since_start'] = (dt - min_date).dt.days

    dt_features_added = ['dt_year','dt_month','dt_quarter',
                          'dt_day_of_week','dt_is_weekend',
                          'dt_hour','dt_days_since_start']
    print(f'Extracted {len(dt_features_added)} datetime features.')
    display(df_scaled[dt_features_added].describe().T)
else:
    print('No datetime column found — Section 6 skipped.')

In [ ]:
if dt_features_added:
    plot_dt = [c for c in dt_features_added if df_scaled[c].nunique() > 1]
    n = len(plot_dt)
    n_cols_p, n_rows_p = 3, (n + 2) // 3
    fig, axes = plt.subplots(n_rows_p, n_cols_p, figsize=(15, 4 * n_rows_p))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, col in enumerate(plot_dt):
        if df_scaled[col].nunique() <= 15:
            df_scaled[col].value_counts().sort_index().plot(
                kind='bar', ax=axes[i], color='teal', edgecolor='white')
        else:
            axes[i].hist(df_scaled[col].dropna(), bins=40,
                         color='teal', edgecolor='white', alpha=0.8)
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Extracted Datetime Features', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 7. Categorical Encoding

Algorithms require numerical inputs — categorical columns must be encoded.
The strategy depends on cardinality:

| Cardinality | Strategy | Rationale |
|-------------|----------|-----------|
| ≤ 10 unique | **One-Hot Encoding** | Creates a binary column per category; interpretable; no ordinal assumption |
| > 10 unique | **Frequency Encoding** | Maps each category to its relative frequency; no dimensionality explosion |
| > 10 unique (if CE available) | **Target Encoding** | Maps to mean of target; captures relationship; risk of leakage — not used for train/test splits without CV |

ID-like columns and `Description` (free-text, > 4000 unique values) are excluded.
The `CustomerID` column (treated as sentinel in Section 3) is kept as-is.

In [ ]:
df_encoded = df_scaled.copy()
encode_log = []

# Columns eligible for encoding
cat_eligible = [
    c for c in CAT_COLS
    if c not in COLS_TO_DROP
    and c != DESC_COL
    and DATE_COL != c
    and df_encoded[c].nunique() >= 2
]

OHE_THRESHOLD = 10  # max unique values for one-hot encoding

# Proxy target for target encoding (highest non-ID numeric col)
proxy_target_col = ('Revenue' if 'Revenue' in df_encoded.columns
                    else (analysis_num[0] if analysis_num else None))

for col in cat_eligible:
    n_unique = df_encoded[col].nunique()

    if n_unique <= OHE_THRESHOLD:
        # One-Hot Encoding
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=False, dtype=int)
        df_encoded = pd.concat([df_encoded.drop(columns=[col]), dummies], axis=1)
        encode_log.append({'column': col, 'strategy': 'One-Hot',
                            'n_unique': n_unique, 'cols_added': list(dummies.columns)})
    else:
        if HAS_CE and proxy_target_col:
            # Target Encoding (mean of proxy target)
            enc = ce.TargetEncoder(cols=[col], smoothing=5)
            enc.fit(df_encoded[col], df_encoded[proxy_target_col])
            df_encoded[f'{col}_te'] = enc.transform(df_encoded[col])
            df_encoded = df_encoded.drop(columns=[col])
            encode_log.append({'column': col, 'strategy': 'Target Encoding',
                                'n_unique': n_unique, 'cols_added': [f'{col}_te']})
        else:
            # Frequency Encoding
            freq_map = df_encoded[col].value_counts(normalize=True)
            df_encoded[f'{col}_freq'] = df_encoded[col].map(freq_map)
            df_encoded = df_encoded.drop(columns=[col])
            encode_log.append({'column': col, 'strategy': 'Frequency Encoding',
                                'n_unique': n_unique, 'cols_added': [f'{col}_freq']})

display(pd.DataFrame(encode_log))
print(f'\nShape after encoding: {df_encoded.shape}')

In [ ]:
print('Sample of encoded dataframe (first 3 rows, all columns):')
display(df_encoded.head(3))

---
## 8. Feature Creation (Domain-Driven)

Raw columns capture individual transaction-level data. E-commerce models
typically need **customer-level** and **order-level** aggregates as features.
We create the following only if the required source columns are detected:

| Feature | Formula | Business meaning |
|---------|---------|------------------|
| `revenue_per_unit` | Revenue / Quantity | Effective unit price; captures bulk discounts |
| `customer_order_count` | nunique(InvoiceNo) per CustomerID | Purchase frequency |
| `customer_avg_order_value` | mean(OrderRevenue) per CustomerID | Average spend per visit |
| `customer_total_revenue` | sum(Revenue) per CustomerID | Monetary value (M in RFM) |
| `customer_recency_days` | max(Date) − reference date | Days since last purchase (R in RFM) |
| `RFM_R`, `RFM_F`, `RFM_M` | Quantile-binned R, F, M scores (1–4) | RFM segment scores |

These features are merged back at the **transaction level** so the row count
is preserved — no aggregation that loses rows.

In [ ]:
df_feat = df_encoded.copy()
feat_log = []

# ── Revenue per unit ─────────────────────────────────────────────────────────
if QTY_COL and 'Revenue' in df_feat.columns:
    safe_qty = df_feat[QTY_COL].replace(0, np.nan)
    df_feat['revenue_per_unit'] = (df_feat['Revenue'] / safe_qty).fillna(0)
    feat_log.append('revenue_per_unit')

# ── Customer-level aggregates ─────────────────────────────────────────────────
if CUSTOMER_COL and CUSTOMER_COL in df_feat.columns:
    cust_df = df.copy()  # use original df for aggregations (pre-transform)
    if QTY_COL:   cust_df = cust_df[cust_df[QTY_COL] > 0]
    if 'Revenue' in cust_df.columns: cust_df = cust_df[cust_df['Revenue'] > 0]

    if INVOICE_COL:
        order_rev = (cust_df.groupby(INVOICE_COL)['Revenue'].sum()
                     .reset_index(name='order_revenue')
                     if 'Revenue' in cust_df.columns else None)

        cust_agg = cust_df.groupby(CUSTOMER_COL).agg(
            customer_order_count=(INVOICE_COL, 'nunique'),
            customer_total_revenue=('Revenue', 'sum') if 'Revenue' in cust_df.columns else (INVOICE_COL, 'count'),
        ).reset_index()

        if 'Revenue' in cust_df.columns:
            inv_rev = cust_df.groupby(INVOICE_COL)['Revenue'].sum().reset_index(name='_inv_rev')
            inv_cust = cust_df[[INVOICE_COL, CUSTOMER_COL]].drop_duplicates()
            inv_rev = inv_rev.merge(inv_cust, on=INVOICE_COL, how='left')
            cust_aov = inv_rev.groupby(CUSTOMER_COL)['_inv_rev'].mean().reset_index(
                name='customer_avg_order_value')
            cust_agg = cust_agg.merge(cust_aov, on=CUSTOMER_COL, how='left')
            feat_log += ['customer_order_count', 'customer_total_revenue', 'customer_avg_order_value']
        else:
            feat_log.append('customer_order_count')

        df_feat = df_feat.merge(cust_agg, on=CUSTOMER_COL, how='left')

    # ── Recency ───────────────────────────────────────────────────────────────
    if DATE_COL and DATE_COL in df.columns:
        ref_date = df[DATE_COL].max()
        last_purchase = (df.groupby(CUSTOMER_COL)[DATE_COL]
                         .max()
                         .reset_index(name='_last_dt'))
        last_purchase['customer_recency_days'] = (
            ref_date - last_purchase['_last_dt']
        ).dt.days
        df_feat = df_feat.merge(
            last_purchase[[CUSTOMER_COL, 'customer_recency_days']],
            on=CUSTOMER_COL, how='left'
        )
        feat_log.append('customer_recency_days')

    # ── RFM scores ────────────────────────────────────────────────────────────
    rfm_cols = {}
    if 'customer_recency_days' in df_feat.columns:
        df_feat['RFM_R'] = pd.qcut(
            df_feat['customer_recency_days'].fillna(df_feat['customer_recency_days'].median()),
            q=4, labels=[4,3,2,1], duplicates='drop'
        ).astype(float)
        rfm_cols['R'] = 'RFM_R'
    if 'customer_order_count' in df_feat.columns:
        df_feat['RFM_F'] = pd.qcut(
            df_feat['customer_order_count'].fillna(1),
            q=4, labels=[1,2,3,4], duplicates='drop'
        ).astype(float)
        rfm_cols['F'] = 'RFM_F'
    if 'customer_total_revenue' in df_feat.columns:
        df_feat['RFM_M'] = pd.qcut(
            df_feat['customer_total_revenue'].fillna(0).clip(lower=0),
            q=4, labels=[1,2,3,4], duplicates='drop'
        ).astype(float)
        rfm_cols['M'] = 'RFM_M'

    if len(rfm_cols) == 3:
        df_feat['RFM_score'] = (
            df_feat['RFM_R'] + df_feat['RFM_F'] + df_feat['RFM_M']
        )
        feat_log += ['RFM_R', 'RFM_F', 'RFM_M', 'RFM_score']
        print('RFM features created.')

print(f'\nNew features added: {feat_log}')

In [ ]:
if feat_log:
    available = [c for c in feat_log if c in df_feat.columns]
    n = min(len(available), 6)
    n_cols_f, n_rows_f = 3, (n + 2) // 3
    fig, axes = plt.subplots(n_rows_f, n_cols_f, figsize=(15, 4 * n_rows_f))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, col in enumerate(available[:n]):
        s = df_feat[col].dropna()
        clipped = s.clip(s.quantile(0.01), s.quantile(0.99))
        if s.nunique() <= 8:
            s.value_counts().sort_index().plot(
                kind='bar', ax=axes[i], color='mediumseagreen', edgecolor='white')
        else:
            axes[i].hist(clipped, bins=50, color='mediumseagreen',
                         edgecolor='white', alpha=0.8)
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribution of Engineered Features', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 9. Feature Selection

Since the modeling goal is not yet defined, we apply three **model-agnostic**
selection methods and report rankings. Final selection will depend on the
algorithm chosen downstream.

| Method | What it removes | When to apply |
|--------|-----------------|---------------|
| **Variance Threshold** | Near-constant features | Always — zero-variance columns add no information |
| **Correlation Filter** | Redundant features (r > 0.90) | Before distance-based models (KNN, clustering, PCA) |
| **Mutual Information** | Ranks features by information content | Any supervised task; use to prioritise features |

We report what **would** be dropped; the actual drop is not applied so the
engineer (you) can override based on the chosen model.

In [ ]:
# Build analysis matrix: numeric, non-null, finite
num_final = df_feat.select_dtypes(include='number').columns.tolist()
# Drop datetime column if it ended up numeric
if DATE_COL in num_final:
    num_final.remove(DATE_COL)

X_analysis = (df_feat[num_final]
               .replace([np.inf, -np.inf], np.nan)
               .fillna(df_feat[num_final].median()))

# Variance threshold
VAR_THRESHOLD = 0.01
vt = VarianceThreshold(threshold=VAR_THRESHOLD)
vt.fit(X_analysis)
low_var_cols = [num_final[i] for i, s in enumerate(vt.variances_) if s <= VAR_THRESHOLD]

var_df = pd.DataFrame({
    'column': num_final,
    'variance': vt.variances_,
    'drop_low_var': ['YES' if c in low_var_cols else 'keep' for c in num_final]
}).set_index('column').sort_values('variance')

print(f'Columns with variance <= {VAR_THRESHOLD}: {low_var_cols}')
display(var_df.head(20))

In [ ]:
corr_matrix = X_analysis.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

CORR_THRESHOLD = 0.90
high_corr_pairs = [
    (col, row_idx, corr_matrix.loc[row_idx, col])
    for col in upper_tri.columns
    for row_idx in upper_tri.index
    if pd.notna(upper_tri.loc[row_idx, col]) and upper_tri.loc[row_idx, col] > CORR_THRESHOLD
]

if high_corr_pairs:
    hc_df = pd.DataFrame(high_corr_pairs, columns=['col_A', 'col_B', 'correlation'])
    hc_df['recommendation'] = 'drop col_B (keep col_A)'
    print(f'Pairs with |r| > {CORR_THRESHOLD} — candidate for removal:')
    display(hc_df)

    # Heatmap of highly correlated subset
    involved = list(set(hc_df['col_A'].tolist() + hc_df['col_B'].tolist()))
    if len(involved) >= 2:
        fig, ax = plt.subplots(figsize=(max(6, len(involved)), max(5, len(involved) - 1)))
        sns.heatmap(X_analysis[involved].corr(), annot=True, fmt='.2f',
                    cmap='coolwarm', center=0, ax=ax)
        ax.set_title(f'High-Correlation Cluster (|r| > {CORR_THRESHOLD})')
        plt.tight_layout()
        plt.show()
else:
    print(f'No feature pairs exceed the {CORR_THRESHOLD} correlation threshold.')

In [ ]:
# Proxy target: most relevant numeric column
proxy_col = ('Revenue' if 'Revenue' in X_analysis.columns
             else X_analysis.columns[0])

X_mi = X_analysis.drop(columns=[proxy_col], errors='ignore')
y_mi = X_analysis[proxy_col]

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_df = (
    pd.DataFrame({'feature': X_mi.columns, 'mutual_info': mi_scores})
    .sort_values('mutual_info', ascending=False)
    .set_index('feature')
)

print(f'Mutual Information scores vs proxy target: {proxy_col}')
display(mi_df)

fig, ax = plt.subplots(figsize=(10, max(4, len(mi_df) * 0.4)))
mi_df['mutual_info'].sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Mutual Information Ranking vs {proxy_col}')
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

In [ ]:
print('FEATURE SELECTION SUMMARY')
print('=' * 60)
print(f'Total numeric features available : {len(num_final)}')
print(f'Low-variance (would drop)        : {len(low_var_cols)}')
print(f'High-correlation pairs flagged   : {len(high_corr_pairs)}')
print(f'\nTop-10 features by mutual info:')
display(mi_df.head(10))
print('\nNote: Final feature selection depends on the chosen model type.')
print('  - Regression/classification: use MI ranking + remove high-corr pairs')
print('  - Clustering: remove highly correlated + scale everything (done)')
print('  - Tree models: variance threshold is sufficient; scale not needed')

---
## 10. Final Engineered Dataset

We assemble the final model-ready dataframe, confirm shape and dtypes,
and save it to disk as `data_engineered.csv` in the `data/processed/` folder.

In [ ]:
# Drop original datetime column and raw categorical columns
# (already replaced by encoded versions)
drop_final = []
if DATE_COL and DATE_COL in df_feat.columns:
    drop_final.append(DATE_COL)

# Drop any remaining object columns (text) that were not encoded
leftover_obj = df_feat.select_dtypes(include='object').columns.tolist()
drop_final += leftover_obj

df_final = df_feat.drop(columns=drop_final, errors='ignore')

# Replace inf values
df_final = df_final.replace([np.inf, -np.inf], np.nan)

# Final null fill with column medians
for col in df_final.select_dtypes(include='number').columns:
    if df_final[col].isna().any():
        df_final[col] = df_final[col].fillna(df_final[col].median())

print(f'Final shape : {df_final.shape[0]:,} rows x {df_final.shape[1]} columns')
print(f'Remaining nulls: {df_final.isna().sum().sum()}')
print(f'Dropped as raw/leftover: {drop_final}')
display(df_final.dtypes.rename('dtype').to_frame())

In [ ]:
display(df_final.head(5))
display(df_final.describe().T)

In [ ]:
# Ensure processed directory exists
processed_dir_candidates = [
    os.path.join('..', 'data', 'processed'),
    os.path.join('ecommerce_analysis', 'data', 'processed'),
    os.path.join('..', '..', 'data', 'processed'),
]
PROCESSED_DIR = next(
    (p for p in processed_dir_candidates if os.path.isdir(os.path.dirname(p))),
    processed_dir_candidates[0]
)
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUT_PATH = os.path.join(PROCESSED_DIR, 'data_engineered.csv')
df_final.to_csv(OUT_PATH, index=False)
print(f'Saved to: {os.path.abspath(OUT_PATH)}')
print(f'Shape   : {df_final.shape}')

---
### Transformation Changelog

| Column / Group | Transformation Applied | Reason |
|---------------|------------------------|--------|
| ID columns (InvoiceNo, StockCode) | **Dropped** | No generalizable signal; unique per transaction |
| Description | **Dropped** | Free-text; 4000+ unique values; needs NLP pipeline |
| CustomerID | **Sentinel fill (-1)** | Missing = guest checkout; meaningful category |
| Numerical — skewed | **Median imputation** | Robust to outliers in skewed distributions |
| Numerical — normal | **Mean imputation** | Mean = median for symmetric distributions |
| Categorical (low null) | **Mode imputation** | Most frequent category is a safe default |
| Categorical (high null) | **'Unknown' category** | Preserves the fact that data is missing |
| Right-skewed + positive cols | **log1p transform** | Compresses heavy right tail; stabilises variance |
| Remaining outlier cols | **Winsorize [p01, p99]** | Caps extremes without removing rows |
| Numerical (scaled) | **StandardScaler / MinMaxScaler / RobustScaler** | Chosen per distribution shape |
| InvoiceDate | **Decomposed** → year, month, quarter, day_of_week, is_weekend, hour, days_since_start | Temporal patterns are model-usable only as numbers |
| Low-card categoricals (≤10) | **One-Hot Encoding** | Binary, interpretable, no ordinal assumption |
| High-card categoricals (>10) | **Target / Frequency Encoding** | Avoids dimensionality explosion |
| New: revenue_per_unit | **Created** = Revenue / Quantity | Captures effective unit price; bulk discount signal |
| New: customer_order_count | **Created** = nunique(InvoiceNo) per customer | Purchase frequency (F in RFM) |
| New: customer_avg_order_value | **Created** = mean order revenue per customer | Avg spend per visit |
| New: customer_total_revenue | **Created** = sum(Revenue) per customer | Monetary value (M in RFM) |
| New: customer_recency_days | **Created** = days since last purchase | Recency (R in RFM) |
| New: RFM_R, RFM_F, RFM_M, RFM_score | **Created** via quartile binning | Customer segment scores for clustering |
| Near-zero variance features | **Flagged** (not yet dropped) | Reported for engineer review |
| Highly correlated pairs (r > 0.90) | **Flagged** (not yet dropped) | Reported for engineer review |
| All features | **MI-ranked** vs Revenue proxy | Priority guide for supervised model feature selection |